# ***Tenno: Data Privacy Act***

## **Qwen-7B Baseline Evaluation**

This notebook evaluates Qwen2.5-7B-Instruct under a Retrieval-Augmented Generation (RAG) setup. For each frozen baseline prompt, the system retrieves the top-k most relevant legal chunks from the prebuilt FAISS index and supplies them as context to the model. The model then generates an answer grounded in retrieved text.


All outputs are saved to Google Drive in JSON format for later aggregation and metric computation (Exact Match, ROUGE-L, Faithfulness, Recall@k).

## **Install Required Packages**

---


In [2]:
!pip install -q transformers accelerate bitsandbytes faiss-cpu sentence-transformers tqdm

## **Mount Drive**

---


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Mounts Google Drive so the notebook can read the frozen prompts and FAISS retrieval artifacts and save model outputs back to Drive

## **Persistent Hugging Face CachePersistent Hugging Face Cache**

---


In [4]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project"
HF_CACHE_DIR = f"{PROJECT_DIR}/Models/hf_cache"  # persistent cache in Drive
os.makedirs(HF_CACHE_DIR, exist_ok=True)

# Force HF/Transformers to use Drive cache
os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR

print("✅ Using persistent HF cache:", HF_CACHE_DIR)

✅ Using persistent HF cache: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Models/hf_cache


This ensures model downloads are cached in Google Drive so you don’t re-download weights every session.

## **Define Paths + Load Prompts and FAISS Artifacts**

---


In [5]:
import os, json
import faiss

PROJECT_DIR = "/content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project"

PROMPTS_FILE = f"{PROJECT_DIR}/Prompts/baseline_prompts_v1.json"
FAISS_INDEX_PATH = f"{PROJECT_DIR}/Chunks/faiss_index.bin"
FAISS_META_PATH = f"{PROJECT_DIR}/Chunks/faiss_metadata.json"

RESULTS_DIR = f"{PROJECT_DIR}/Results"
os.makedirs(RESULTS_DIR, exist_ok=True)

OUTPUT_PATH = f"{RESULTS_DIR}/baseline_outputs_qwen.json"

# Load prompts
with open(PROMPTS_FILE, "r", encoding="utf-8") as f:
    prompts = json.load(f)

# Load FAISS index
index = faiss.read_index(FAISS_INDEX_PATH)

# Load metadata (must align with index vector order)
with open(FAISS_META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

print("✅ Prompts loaded:", len(prompts))
print("✅ FAISS vectors:", index.ntotal)
print("✅ Metadata rows:", len(meta))
print("✅ Output path:", OUTPUT_PATH)

✅ Prompts loaded: 10
✅ FAISS vectors: 198
✅ Metadata rows: 198
✅ Output path: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Results/baseline_outputs_qwen.json


This loads the frozen prompt list, the saved FAISS vector index, and the metadata mapping from vector IDs back to chunk text and source information. These three components form the retrieval backbone of the RAG evaluation.

## **Load SentenceTransformer for Query Embedding**

---


In [6]:
from sentence_transformers import SentenceTransformer

embed_model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name)

print(f"✅ Query embedder loaded: {embed_model_name}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Query embedder loaded: all-MiniLM-L6-v2


FAISS retrieval requires embedding the user query (the prompt question) into a vector. We reuse the same embedding model used to build the FAISS index to ensure compatible vector representations.

## **Retrieval Helper (Top-k Chunks)**

---


In [7]:
import numpy as np

TOP_K = 3

def retrieve_top_k_chunks(query: str, top_k: int = TOP_K):
    q_emb = embedder.encode([query], convert_to_numpy=True)
    q_emb = q_emb.astype(np.float32)

    distances, indices = index.search(q_emb, top_k)
    idxs = indices[0].tolist()

    retrieved = []
    for i in idxs:
        row = meta[i]
        retrieved.append({
            "faiss_id": i,
            "text": row.get("text", ""),
            "source": row.get("source", ""),
            "page_number": row.get("page_number", None),
            "distance": float(distances[0][idxs.index(i)])
        })
    return retrieved

This function embeds the prompt question, retrieves the top-k closest chunk vectors from FAISS, and returns the corresponding chunk text and metadata.

## **Load Qwen-7B Model (4-bit)**

---


In [12]:
import os, gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading Qwen tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE_DIR)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Cleanup before heavy load
gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Loading Qwen model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=HF_CACHE_DIR,
    device_map="auto",
    quantization_config=bnb_config,
    low_cpu_mem_usage=True
)
model.eval()
print("✅ Qwen loaded in 4-bit.")

Loading Qwen tokenizer...
Loading Qwen model (4-bit)...


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

✅ Qwen loaded in 4-bit.


This loads Qwen2.5-7B-Instruct using Hugging Face Transformers. device_map="auto" places the model on available GPU resources. The model will generate answers using retrieved legal context.

## **Build Model Input (Context + Question)**

---


In [13]:
def build_rag_input(retrieved_chunks, question: str) -> str:
    context_blocks = []
    for c in retrieved_chunks:
        src = c["source"]
        pg = c["page_number"]
        txt = c["text"]
        context_blocks.append(f"[Source: {src} | Page: {pg}]\n{txt}")

    context = "\n\n---\n\n".join(context_blocks)

    return (
        "You are a legal assistant. Answer the question using ONLY the retrieved context.\n\n"
        f"Retrieved Context (Top-{TOP_K}):\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )

This formats the retrieved chunk evidence into a single context block, then appends the prompt question.

## **Deterministic Generation Helper**

---


In [14]:
@torch.inference_mode()
def generate_answer(input_text: str, max_new_tokens: int = 256) -> str:
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    out_ids = model.generate(
        **inputs,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    return decoded[len(input_text):].strip()

This generates answers using deterministic decoding (temperature=0, do_sample=False) to ensure reproducibility across runs. The function returns only the generated portion of the output.

## **Run All Prompts and Save Outputs**

---


In [15]:
from tqdm import tqdm
import time

results = []

for p in tqdm(prompts, desc="Running Qwen RAG prompts"):
    q = p["question"]

    retrieved = retrieve_top_k_chunks(q, top_k=TOP_K)
    input_text = build_rag_input(retrieved, q)
    answer = generate_answer(input_text, max_new_tokens=256)

    results.append({
        "model_id": MODEL_ID,
        "prompt_id": p.get("prompt_id"),
        "title": p.get("title"),
        "method": p.get("method"),
        "condition": p.get("condition"),
        "dataset_file": p.get("dataset_file"),
        "section_index": p.get("section_index"),
        "question": q,
        "retrieved_chunks": retrieved,
        "answer": answer,
        "expected_output": p.get("expected_output")
    })

# Save
with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print("✅ Saved outputs to:", OUTPUT_PATH)

Running Qwen RAG prompts: 100%|██████████| 10/10 [04:39<00:00, 27.98s/it]


✅ Saved outputs to: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Results/baseline_outputs_qwen.json


This loop runs all baseline prompts through the full RAG pipeline: query embedding → FAISS retrieval → context construction → Qwen generation. Outputs are saved to a JSON file containing both retrieved evidence and generated responses for later scoring.

## **Display Outputs**

---


In [16]:
from pprint import pprint
import json

RESULTS_PATH = f"{PROJECT_DIR}/Results/baseline_outputs_qwen.json"

# Load results
with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    results = json.load(f)

print(f"\n✅ Loaded {len(results)} Qwen outputs\n")

# Display brief preview
for item in results:
    print("=" * 80)
    print(f"[Prompt {item['prompt_id']}] ({item['method']})")
    print("-" * 80)
    print("Q:", item["question"])
    print("\nA:", item["answer"][:400])
    print("\n")


✅ Loaded 10 Qwen outputs

[Prompt 1] (Zero-Shot)
--------------------------------------------------------------------------------
Q: List all the conditions under which the processing of personal information is considered lawful according to this section.

A: The processing of personal information is considered lawful if it meets at least one of the following conditions:
(a) the data subject has given his or her consent;
(b) the processing of personal information is necessary and is related to the fulfillment of a contract with the data subject or in order to take steps at the request of the data subject prior to entering into a contract;
(c) the proce


[Prompt 2] (Zero-Shot)
--------------------------------------------------------------------------------
Q: What are the penalties for unauthorized disclosure of personal information and sensitive personal information under this section?

A: The penalties for unauthorized disclosure of personal information are imprisonment ranging from

This prints a compact preview of each generated answer to verify the notebook produced reasonable outputs before moving to the next model.